In [1]:
import pandas as pd
import sklearn

In [51]:
# Load movie data
csv_file = "./enhanced_box_office_data(2000-2024).csv"
movie_data = pd.read_csv(csv_file)

# Remove rows with null values (usually, specific rows will be missing many of the values, as noted here: 
#   https://www.kaggle.com/code/celalngr/movie-box-office-data-analysis-and-preprocessing/notebook#Visualization-of-Missing-Data)
movie_data.dropna(inplace=True)

# =================================================================================================
# RATINGS
# =================================================================================================

# Remove "/10" from ratings and convert to float
movie_data['Rating'] = movie_data['Rating'].str.replace('/10', '', regex=True).str.strip()
movie_data['Rating'] = movie_data['Rating'].astype(float)

# =================================================================================================
# GENRES
# =================================================================================================

# Get a list of unique genres and map to index ids
movie_data["Genres"] = movie_data["Genres"].astype(str).str.split(pat=',').apply(lambda list: [genre.strip() for genre in list])
genres = set(movie_data["Genres"].explode())
genre_to_idx = dict(zip(sorted(genres), range(len(genres))))
idx_to_genre = dict(zip(range(len(genres)), sorted(genres)))

# Convert genre lists into multi-hot vectors (0/1 boolean vectors for categories)
genre_dummies = movie_data["Genres"].explode().str.get_dummies().groupby(level=0).max()
genre_dummies = genre_dummies[sorted(genre_dummies.columns)]
movie_data["Genres"] = genre_dummies.values.tolist()

# Display useful variables
# print(f"Genre set       : {genres}")
# print(f"Genre-index map : {genre_to_idx}")
# print(f"Index-genre map : {idx_to_genre}")

# =================================================================================================
# PRODUCTION COUNTRIES
# =================================================================================================
top_production_country_count = 20   # Used to simplify problem by lowering dimensionality-- In other words, only create categories for the top n countries
placeholder = "Other"               # Used to replace country names not in the top n countries

# Get a list of top `top_production_country_count` unique countries
movie_data["Production_Countries"] = movie_data["Production_Countries"].astype(str).str.split(pat=',').apply(lambda list: [country.strip() for country in list])
top_countries = movie_data["Production_Countries"].explode().value_counts().head(top_production_country_count)
countries = set(top_countries.index)

# Map top countries to index ids
idx_to_country = dict(zip(range(len(top_countries.index)), list(top_countries.index)))
country_to_idx = dict(zip(list(top_countries.index), range(len(countries))))

# Handle placeholder cases
countries.add(placeholder)
idx_to_country[top_production_country_count] = placeholder
country_to_idx[placeholder] = top_production_country_count

# Convert country lists into multi-hot vectors
country_dummies = movie_data["Production_Countries"].explode()
country_dummies = country_dummies.where(country_dummies.isin(countries), other=placeholder)
country_dummies = country_dummies.str.get_dummies().groupby(level=0).max()
country_dummies = country_dummies[sorted(country_dummies.columns, key=lambda x: country_to_idx[f"{x}"])]
movie_data["Production_Countries"] = country_dummies.values.tolist()

# # Display useful variables
# print(f"Production Country set : {countries}")
# print(f"Country-index map      : {country_to_idx}")
# print(f"Index-country map      : {idx_to_country}")

# =================================================================================================
# FINAL TABLE
# =================================================================================================
display(movie_data)

,Rank,Release Group,$Worldwide,$Domestic,Domestic %,$Foreign,Foreign %,Year,Genres,Rating,Vote_Count,Original_Language,Production_Countries
0,1,Mission: Impossible II,546388108.0,215409889.0,39.4,330978219.0,60.6,2000,"[1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",6.126,6741.0,en,"[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
1,2,Gladiator,460583960.0,187705427.0,40.8,272878533.0,59.2,2000,"[1, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, ...",8.217,19032.0,en,"[1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
2,3,Cast Away,429632142.0,233632142.0,54.4,196000000.0,45.6,2000,"[0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, ...",7.663,11403.0,en,"[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
3,4,What Women Want,374111707.0,182811707.0,48.9,191300000.0,51.1,2000,"[0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, ...",6.450,3944.0,en,"[1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
4,5,Dinosaur,349822765.0,137748063.0,39.4,212074702.0,60.6,2000,"[0, 1, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, ...",6.544,2530.0,en,"[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...
4995,196,Devara Part 1,7361414.0,5600000.0,76.1,1761414.0,23.9,2024,"[1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, ...",7.000,30.0,te,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, ..."
4996,197,Kolpaçino 4 4'lük,7343114.0,0.0,0.0,7343114.0,100.0,2024,"[0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",4.000,4.0,tr,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
4997,198,Lim Young Woong: Im Hero the Stadium,7305588.0,0.0,0.0,7305588.0,100.0,2024,"[0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, ...",0.000,0.0,ko,"[0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, ..."
4998,199,Yolo,7241561.0,2001584.0,27.6,5239977.0,72.4,2024,"[1, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, ...",6.300,70.0,zh,"[0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
